# VisionFlow Benchmarks

Measures speed and accuracy of YOLO11n across PyTorch, ONNX Runtime, and TensorRT (FP16 / INT8), plus multi-stream detection + tracking throughput.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/Altair098/visionflow.git
%cd visionflow
!pip install -q -r requirements.txt
!pip install -q tensorrt

## 1. Test video

Upload any video with people walking (e.g. a free "people walking" or "shopping mall" clip from pexels.com, 720p or 1080p).

In [ ]:
from google.colab import files
import shutil, os

uploaded = files.upload()
name = next(iter(uploaded))
os.makedirs("data/samples", exist_ok=True)
shutil.move(name, "data/samples/test.mp4")
VIDEO = "data/samples/test.mp4"

## 2. COCO val2017 for mAP

5,000 held-out images (about 1 GB). Gives honest accuracy numbers for each backend.

In [ ]:
%%bash
mkdir -p /content/datasets && cd /content/datasets
if [ ! -d coco/images/val2017 ]; then
  wget -q https://github.com/ultralytics/assets/releases/download/v0.0.0/coco2017labels.zip
  unzip -q coco2017labels.zip && rm coco2017labels.zip
  mkdir -p coco/images && cd coco/images
  wget -q http://images.cocodataset.org/zips/val2017.zip
  unzip -q val2017.zip && rm val2017.zip
fi
ls /content/datasets/coco/images/val2017 | wc -l

In [ ]:
import pathlib
import ultralytics
import yaml

coco = yaml.safe_load(open(pathlib.Path(ultralytics.__file__).parent / "cfg/datasets/coco.yaml"))
cfg = {"path": "/content/datasets/coco", "train": "val2017.txt", "val": "val2017.txt", "names": coco["names"]}
yaml.safe_dump(cfg, open("coco_val.yaml", "w"))

## 3. Export models

In [ ]:
!python scripts/export.py --weights yolo11n.pt --format onnx
!python scripts/export.py --weights yolo11n.pt --format engine --half
!mv yolo11n.engine yolo11n_fp16.engine
!python scripts/export.py --weights yolo11n.pt --format engine --int8 --data coco_val.yaml
!mv yolo11n.engine yolo11n_int8.engine
!ls -lh yolo11n*

## 4. Single-stream speed and accuracy

In [ ]:
import json, subprocess

runs = [
    ("PyTorch", "FP32", "yolo11n.pt"),
    ("ONNX Runtime", "FP32", "yolo11n.onnx"),
    ("TensorRT", "FP16", "yolo11n_fp16.engine"),
    ("TensorRT", "INT8", "yolo11n_int8.engine"),
]

results = []
for backend, precision, weights in runs:
    out = subprocess.run(
        ["python", "scripts/benchmark.py", "--weights", weights, "--source", VIDEO,
         "--frames", "500", "--data", "coco_val.yaml"],
        capture_output=True, text=True,
    ).stdout
    r = json.loads(out[out.index("{"):])
    r.update(backend=backend, precision=precision)
    results.append(r)
    print(backend, precision, r)

## 5. Multi-stream detection + tracking (TensorRT FP16)

In [ ]:
!python scripts/multistream_benchmark.py --weights yolo11n_fp16.engine --source {VIDEO} --streams 1 2 4 8 12 16 --frames 300 --target-fps 15 | tee multistream.json

## 6. Results table for the README

In [ ]:
base = results[0]["fps"]
print("| Model | Backend | Precision | FPS | Latency p50 (ms) | Latency p95 (ms) | mAP50-95 | Speedup |")
print("|---|---|---|---|---|---|---|---|")
for r in results:
    print(f"| YOLO11n | {r['backend']} | {r['precision']} | {r['fps']} | {r['latency_ms_p50']} | "
          f"{r['latency_ms_p95']} | {r['map50_95']} | {r['fps'] / base:.1f}x |")

ms = json.loads(open("multistream.json").read()[open("multistream.json").read().index("["):])
print()
print("| Streams | Per-stream FPS | Total FPS | Real-time (>=15 FPS) |")
print("|---|---|---|---|")
for m in ms:
    print(f"| {m['streams']} | {m['per_stream_fps']} | {m['total_fps']} | {'yes' if m['realtime'] else 'no'} |")